[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# A Product Catalog &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell and the second loads a twenty thousand row catalog and
binds the model to it. Run both first, then any task in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pydantic
import pymongo
from beanie import Document, init_beanie
from beanie.operators import In
from pymongo import AsyncMongoClient, IndexModel

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

CATALOG = "catalog"                                                 # one place, spelled once
SIZE = 200_000                                                      # big enough for explain to matter


def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def plan(cursor):
    """How the server answered, which is the only honest way to judge an index."""
    explained = cursor.explain()
    winner = explained["queryPlanner"]["winningPlan"]
    stats = explained["executionStats"]
    return {"stage": winner.get("inputStage", winner).get("stage"),
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


def catalog_rows(size=None):
    """The rows to load, built once from a fixed seed so every run of this notebook agrees."""
    size = SIZE if size is None else size
    random.seed(1)
    kinds = ["laptop", "monitor", "keyboard", "mouse", "cable"]
    makers = ["Aster", "Belden", "Corvid", "Dalgo"]
    return [{"sku": f"SKU-{number:07d}",
             "name": f"{makers[number % 4]} {kinds[number % 5]} {number}",
             "maker": makers[number % 4],
             "kind": kinds[number % 5],
             "price": round(random.uniform(5, 2000), 2),
             "stock": random.randint(0, 500)}
            for number in range(size)]


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products (the guide's own, untouched by this notebook)")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  200000 products (the guide's own, untouched by this notebook)
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 200000


In [2]:
class Product(Document):
    sku: str
    name: str
    maker: str
    kind: str
    price: float
    stock: int = 0

    class Settings:
        name = CATALOG


client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop[CATALOG].drop()
shop[CATALOG].insert_many(catalog_rows(20_000))

async_client = AsyncMongoClient(URI)
await init_beanie(database=async_client.get_default_database(), document_models=[Product])
print("ready:", shop[CATALOG].count_documents({}), "documents")


ready: 20000 documents


**1.** Rows in, with PyMongo.


In [3]:
shop.answers.drop()
rows = catalog_rows(10_000)

for first in range(0, len(rows), 5_000):
    shop.answers.insert_many([dict(row) for row in rows[first:first + 5_000]])

print("loaded:", shop.answers.count_documents({}), "documents")
print("one of them:", shop.answers.find_one({}, {"_id": 0, "sku": 1, "kind": 1, "price": 1}))


loaded: 10000 documents
one of them: {'sku': 'SKU-0000000', 'kind': 'laptop', 'price': 273.06}


Plain dictionaries, in batches. Nothing validated them, which is what makes it fast and is the trade
the capstone is about.


**2.** An index, measured.


In [4]:
for index in list(shop[CATALOG].list_indexes()):
    if index["name"] != "_id_":
        shop[CATALOG].drop_index(index["name"])

query = {"maker": "Corvid"}
print("before:", plan(shop[CATALOG].find(query)))

shop[CATALOG].create_index("maker", name="maker_1")
print("after: ", plan(shop[CATALOG].find(query)))


before: {'stage': 'COLLSCAN', 'documents': 20000, 'returned': 5000}
after:  {'stage': 'IXSCAN', 'documents': 5000, 'returned': 5000}


A collection scan of twenty thousand documents becomes an index scan reading exactly what it
returns. The query did not change.


**3.** The same document, as a model.


In [5]:
raw = shop[CATALOG].find_one({}, {"_id": 0})
product = await Product.find_one(Product.sku == raw["sku"])

print("as a dictionary:", {k: raw[k] for k in ("sku", "kind", "price")})
print("as a model:     ", product.sku, "|", product.kind, "|", product.price)
print("and the price is a real float:", product.price + 0.01 != product.price)


as a dictionary: {'sku': 'SKU-0000000', 'kind': 'laptop', 'price': 273.06}
as a model:      SKU-0000000 | laptop | 273.06
and the price is a real float: True


PyMongo wrote it and Beanie read it. Nothing converted between them, because a collection is not
owned by a library.


**4.** The cheapest three of a kind.


In [6]:
cheapest = await (Product.find(Product.kind == "monitor")
                         .sort(Product.price).limit(3).to_list())

for product in cheapest:
    print(f"  {product.sku}  {product.price:8.2f}  {product.maker}")


  SKU-0008761      5.44  Belden
  SKU-0002596      5.44  Aster
  SKU-0015006      6.61  Corvid


Every field reference is checked against the model, so a typo in the filter or the sort is an
`AttributeError` rather than a query that quietly matches nothing.


**5.** A report, typed.


In [7]:
class ByMaker(pydantic.BaseModel):
    maker: str
    products: int
    average: float


pipeline = [
    {"$group": {"_id": "$maker", "products": {"$sum": 1}, "average": {"$avg": "$price"}}},
    {"$project": {"_id": 0, "maker": "$_id", "products": 1,
                  "average": {"$round": ["$average", 2]}}},
    {"$sort": {"maker": 1}},
]

for row in await Product.aggregate(pipeline, projection_model=ByMaker).to_list():
    print(f"  {row.maker:8} {row.products:6} products  average {row.average:8.2f}")


  Aster      5000 products  average  1017.59
  Belden     5000 products  average   997.88
  Corvid     5000 products  average  1020.83
  Dalgo      5000 products  average   994.50


The `$project` renames `_id` to `maker` because the model has a field of that name. Those names are
a contract written in two places, and Pydantic checks it.


**6.** Asking for an index twice.


In [8]:
first = shop[CATALOG].create_index([("kind", 1), ("price", 1)], name="kind_price")
again = shop[CATALOG].create_index([("kind", 1), ("price", 1)], name="kind_price")

print("first call: ", first)
print("second call:", again, "| the same name, and nothing happened")
print("indexes:", sorted(index["name"] for index in shop[CATALOG].list_indexes()))

shop.answers.drop()
shop[CATALOG].drop()
client.close()
await async_client.close()


first call:  kind_price
second call: kind_price | the same name, and nothing happened
indexes: ['_id_', 'kind_price', 'maker_1']


That is why the same `create_index` can live in the loader and in the model's `Settings` without
conflict, and why it is safe to call at startup: an index that already exists, with the same
options, is a no-op.


---

&#8592; **Back to:** [A Product Catalog](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/16-a-product-catalog.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
